# Watch Store Location Success Prediction
**Machine Learning Decision-Support System for Retail Store Expansion**

> **Academic Disclosure Notice:**  
> Synthetic dataset created for academic machine-learning practice. It does not represent actual watch-store businesses or real-world locations.

---

## 1. Project Introduction
Expanding a retail brand by opening a physical store involves substantial capital investment in lease agreements, store fitting, inventory, and staffing. Choosing an unsuitable site leads to significant financial losses.

This project implements a machine learning binary classification pipeline using **Python, Pandas, and Scikit-learn** to predict whether a proposed retail location is likely to be **Profitable (1)** or **Not Profitable (0)**, while estimating the dynamic **Probability of Profitability (%)** using `predict_proba()`.

## 2. Business Problem
A retail business owner wants an objective, data-driven decision-support tool to evaluate potential store locations before committing capital.

**Goal:**  
Build a classification model using location, customer demographic, competitive, and cost factors to:
1. Classify proposed locations as **Profitable (1)** or **Not Profitable (0)**.
2. Output a dynamic **Probability of Profitability (XX.XX%)**.
3. Enable **Location Comparison** and **What-If Sensitivity Analysis** to optimize expansion site selection.

## 3. Import Libraries

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path for modular imports
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from generate_dataset import generate_watch_store_dataset
from preprocess import prepare_train_test_data
from train_eval import train_and_evaluate_models
from predict import (
    predict_store_success,
    compare_locations,
    sensitivity_analysis,
    financial_investment_calculator,
    generate_business_summary
)

# Plotting style setup
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 10
print("Libraries successfully imported.")

Libraries successfully imported.


## 4. Generate / Load Dataset
We load the 5,000-record dataset generated by `src/generate_dataset.py`.

In [2]:
dataset_path = os.path.join('..', 'data', 'watch_store_locations.csv')

if not os.path.exists(dataset_path):
    df = generate_watch_store_dataset(num_samples=5000, random_seed=42)
    os.makedirs(os.path.dirname(dataset_path), exist_ok=True)
    df.to_csv(dataset_path, index=False)
    print("Generated and saved new dataset.")
else:
    df = pd.read_csv(dataset_path)
    print(f"Dataset successfully loaded from {dataset_path}.")

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset successfully loaded from ..\data\watch_store_locations.csv.
Dataset Shape: 5000 rows, 13 columns


## 5. Dataset Overview
Displaying the first 5 records and column statistics.

In [3]:
df.head()

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   population                   5000 non-null   int64  
 1   average_monthly_income       5000 non-null   float64
 2   daily_foot_traffic           5000 non-null   int64  
 3   nearby_competitors           5000 non-null   int64  
 4   monthly_rent                 5000 non-null   float64
 5   distance_to_mall_km          5000 non-null   float64
 6   nearby_retail_stores         5000 non-null   int64  
 7   estimated_monthly_customers  5000 non-null   int64  
 8   average_purchase_value       5000 non-null   float64
 9   monthly_operating_cost       5000 non-null   float64
 10  local_demand_score           5000 non-null   float64
 11  target_age_group_score       5000 non-null   float64
 12  profitable                   5000 non-null   int64  
dtypes: float64(7), int

In [5]:
df.describe().T

## 6. Data Quality Check
Checking for missing values and duplicate rows across all features.

In [6]:
print("Missing Values Check per Column:")
print(df.isnull().sum())
print(f"\nDuplicate Rows Check: {df.duplicated().sum()}")

Missing Values Check per Column:
population                     0
average_monthly_income         0
daily_foot_traffic             0
nearby_competitors             0
monthly_rent                   0
distance_to_mall_km            0
nearby_retail_stores           0
estimated_monthly_customers    0
average_purchase_value         0
monthly_operating_cost         0
local_demand_score             0
target_age_group_score         0
profitable                     0
dtype: int64

Duplicate Rows Check: 0


## 7. Exploratory Data Analysis (EDA)
Visualizing distributions of key location attributes and feature correlations.

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['daily_foot_traffic'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Daily Foot Traffic')

sns.histplot(df['monthly_rent'], kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Distribution of Monthly Rent ($)')

sns.boxplot(x='profitable', y='average_monthly_income', hue='profitable', data=df, ax=axes[1, 0], palette='Set2', legend=False)
axes[1, 0].set_title('Income by Profitability Status')
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_xticklabels(['Not Profitable', 'Profitable'])

sns.boxplot(x='profitable', y='monthly_operating_cost', hue='profitable', data=df, ax=axes[1, 1], palette='Set2', legend=False)
axes[1, 1].set_title('Operating Cost by Profitability Status')
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(['Not Profitable', 'Profitable'])

plt.tight_layout()
plt.show()

In [8]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

## 8. Target Distribution
Examining class balance for `profitable` (0 = Not Profitable, 1 = Profitable).

In [9]:
target_counts = df['profitable'].value_counts()
target_props = df['profitable'].value_counts(normalize=True) * 100

print("Target Class Counts:")
print(target_counts)
print("\nTarget Class Proportions (%):")
print(target_props)

plt.figure(figsize=(6, 4))
sns.countplot(x='profitable', data=df, palette=['#e74c3c', '#2ecc71'])
plt.xticks([0, 1], ['Not Profitable (0)', 'Profitable (1)'])
plt.title("Target Class Distribution")
plt.ylabel("Count")
plt.show()

Target Class Counts:
profitable
1    2934
0    2066
Name: count, dtype: int64

Target Class Proportions (%):
profitable
1    58.68
0    41.32
Name: proportion, dtype: float64


## 9. Feature Preparation
Separating feature matrix `X` (12 location attributes) and target vector `y`. Scaling numerical features using `StandardScaler` fitted ONLY on training data for Logistic Regression.

In [10]:
data_dict = prepare_train_test_data(csv_path=dataset_path)
print("Features prepared and scaled successfully.")
print(f"Feature count: {len(data_dict['feature_names'])}")

Features prepared and scaled successfully.
Feature count: 12


## 10. Train / Test Split
Applying an 80/20 stratified split (`test_size=0.2, random_state=42, stratify=y`) to maintain target proportions across splits.

In [11]:
print(f"Training Set Size: {data_dict['X_train'].shape[0]} samples")
print(f"Testing Set Size: {data_dict['X_test'].shape[0]} samples")

Training Set Size: 4000 samples
Testing Set Size: 1000 samples


## 11. Logistic Regression Model
Training `LogisticRegression` baseline on scaled training features.

In [12]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(data_dict['X_train_scaled'], data_dict['y_train'])
lr_preds = lr_model.predict(data_dict['X_test_scaled'])
lr_probs = lr_model.predict_proba(data_dict['X_test_scaled'])[:, 1]
print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


## 12. Random Forest Classifier
Training non-linear `RandomForestClassifier` (100 estimators) on unscaled training features.

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(data_dict['X_train'], data_dict['y_train'])
rf_preds = rf_model.predict(data_dict['X_test'])
rf_probs = rf_model.predict_proba(data_dict['X_test'])[:, 1]
print("Random Forest Classifier trained successfully.")

Random Forest Classifier trained successfully.


## 13. Model Evaluation
Evaluating Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrices on the held-out test set.

In [14]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_lr = confusion_matrix(data_dict['y_test'], lr_preds)
cm_rf = confusion_matrix(data_dict['y_test'], rf_preds)

ConfusionMatrixDisplay(cm_lr, display_labels=['Not Profitable', 'Profitable']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title("Logistic Regression Confusion Matrix")

ConfusionMatrixDisplay(cm_rf, display_labels=['Not Profitable', 'Profitable']).plot(ax=axes[1], cmap='Greens')
axes[1].set_title("Random Forest Confusion Matrix")

plt.tight_layout()
plt.show()

## 14. Model Comparison
Comparing empirical performance across models on the test set and selecting the best model based on ROC-AUC.

In [15]:
eval_results = train_and_evaluate_models()
print(eval_results['metrics_df'].to_string(index=False))
print(f"\nMODEL SELECTION DECISION:\n{eval_results['selected_reason']}")

                   Model Accuracy Precision Recall F1-Score ROC-AUC
     Logistic Regression   0.9110    0.9164 0.9336   0.9249  0.9729
Random Forest Classifier   0.9370    0.9411 0.9523   0.9467  0.9850

MODEL SELECTION DECISION:
Random Forest Classifier was selected for inference because it achieved the higher ROC-AUC (0.9850 vs 0.9729) on this test evaluation.


## 15. Probability Prediction (`predict_proba`)
Demonstrating dynamic probability estimation for a candidate store location using `predict_store_success()`.

In [16]:
sample_location = {
    'population': 180000,
    'average_monthly_income': 6500.0,
    'daily_foot_traffic': 14000,
    'nearby_competitors': 3,
    'monthly_rent': 6500.0,
    'distance_to_mall_km': 1.2,
    'nearby_retail_stores': 45,
    'estimated_monthly_customers': 1200,
    'average_purchase_value': 320.0,
    'monthly_operating_cost': 7500.0,
    'local_demand_score': 8.2,
    'target_age_group_score': 7.5
}

res = predict_store_success(sample_location)
print(f"Prediction: {res['prediction']}")
print(f"Probability of Profitability: {res['probability_percent']}%")

Prediction: PROFITABLE
Probability of Profitability: 93.0%


## 16. Example Store Locations
Defining three candidate expansion sites for multi-site evaluation.

In [17]:
sites = {
    'Downtown Commercial Center': sample_location,
    'Suburban Strip Mall': {
        'population': 75000,
        'average_monthly_income': 4200.0,
        'daily_foot_traffic': 4500,
        'nearby_competitors': 6,
        'monthly_rent': 11000.0,
        'distance_to_mall_km': 5.5,
        'nearby_retail_stores': 20,
        'estimated_monthly_customers': 400,
        'average_purchase_value': 180.0,
        'monthly_operating_cost': 9000.0,
        'local_demand_score': 4.5,
        'target_age_group_score': 5.0
    },
    'High-Street Pedestrian Zone': {
        'population': 250000,
        'average_monthly_income': 8500.0,
        'daily_foot_traffic': 22000,
        'nearby_competitors': 2,
        'monthly_rent': 8500.0,
        'distance_to_mall_km': 0.5,
        'nearby_retail_stores': 90,
        'estimated_monthly_customers': 1800,
        'average_purchase_value': 450.0,
        'monthly_operating_cost': 8000.0,
        'local_demand_score': 9.1,
        'target_age_group_score': 8.8
    }
}
print("Sample expansion locations defined.")

Sample expansion locations defined.


## 17. Location Comparison
Ranking candidate expansion sites by model-estimated profitability probability.

In [18]:
comp_table = compare_locations(sites)
print("--- LOCATIONS RANKED BY MODEL-ESTIMATED PROFITABILITY PROBABILITY ---")
print(comp_table.to_string(index=False))

--- LOCATIONS RANKED BY MODEL-ESTIMATED PROFITABILITY PROBABILITY ---
                   Location     Prediction  Probability (%)  Rent ($)  Foot Traffic  Competitors
High-Street Pedestrian Zone     PROFITABLE             99.0    8500.0         22000            2
 Downtown Commercial Center     PROFITABLE             93.0    6500.0         14000            3
        Suburban Strip Mall NOT PROFITABLE              4.0   11000.0          4500            6


## 18. What-If / Sensitivity Analysis
Dynamically evaluating how varying lease rent impacts profitability probability.

In [19]:
rent_variations = [4000, 6500, 9000, 12000, 15000]
sens_df = sensitivity_analysis(sample_location, 'monthly_rent', rent_variations)
print("--- WHAT-IF SENSITIVITY ANALYSIS (MONTHLY RENT) ---")
print(sens_df.to_string(index=False))

--- WHAT-IF SENSITIVITY ANALYSIS (MONTHLY RENT) ---
 monthly_rent Prediction  Probability (%)
         4000 PROFITABLE             97.0
         6500 PROFITABLE             93.0
         9000 PROFITABLE             85.0
        12000 PROFITABLE             73.0
        15000 PROFITABLE             66.0


## 19. Financial Investment Calculation
Computing separate post-prediction financial heuristics (Revenue, Costs, Profit, Break-Even timeline) using a 10% (0.10) gross retail margin assumption consistent with baseline synthetic economics.

> **Note:** The financial calculator is a separate business analysis component and is NOT used as an input feature for ML model training.

In [20]:
fin_calc = financial_investment_calculator(sample_location)
print("--- FINANCIAL INVESTMENT ESTIMATES (POST-PREDICTION HEURISTIC) ---")
for k, v in fin_calc.items():
    print(f"{k}: {v}")

--- FINANCIAL INVESTMENT ESTIMATES (POST-PREDICTION HEURISTIC) ---
est_monthly_revenue: 38400.0
total_monthly_cost: 14000.0
est_monthly_net_profit: 24400.0
initial_setup_cost: 150000
breakeven_months: 6.1


## 20. Business Summary
Generating an executive assessment report with threshold-based qualitative heuristics and disclaimer notices.

In [21]:
summary = generate_business_summary("Downtown Commercial Center", sample_location)
print(summary)

LOCATION ASSESSMENT: DOWNTOWN COMMERCIAL CENTER
Model Prediction: PROFITABLE
Probability of Profitability: 93.0%
Model Selected: Random Forest Classifier

QUALITATIVE SITE METRICS (Business Threshold Heuristics):
- Daily Foot Traffic: 14000 (High)
- Nearby Competitors: 3 (Low)
- Monthly Rent: $6,500.00 (Moderate)
- Local Demand Score: 8.2/10 (Strong)

SEPARATE FINANCIAL HEURISTICS (Academic Estimate):
- Projected Monthly Revenue: $38,400.00
- Total Monthly Operating Cost: $14,000.00
- Estimated Monthly Net Profit: $24,400.00
- Estimated Break-Even Period: 6.1 months

Disclaimer: Both the ML model probability and financial calculator are academic decision-support estimates and do not guarantee real-world store profitability.


## 21. Limitations
1. **Synthetic Data:** Created for academic practice and machine-learning methodology demonstration.
2. **Static Assumptions:** Does not capture dynamic inflation rates or multi-year competitor entry.
3. **Uncaptured Features:** Storefront visibility, parking availability, and brand strength are not included in the tabular attributes.

## 22. Conclusion
- Built an end-to-end binary classification pipeline predicting watch store location profitability.
- Achieved ~93.70% test accuracy and 0.9850 ROC-AUC using Random Forest Classifier.
- Integrated dynamic probability scores (`predict_proba()`), location ranking, and sensitivity analysis into a business decision-support tool.